# Generate simulations

This notebook creates the model data for all figure notebooks. It needs no input files. It saves new simulation tables in generated/. The full screen has not been run in this package.

## Parameters

Edit the next cell to change the final run settings.

In [ ]:
from pathlib import Path
import json
from collections import deque
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp, trapezoid

PROFILE = 'final_10000'
SEED = 1993
SAVE = True
OUTPUT_DIR = Path('generated')
RICHNESS = [10, 25, 50, 100, 125]
DEGREES = np.linspace(0.1, 10, 101)
STRENGTHS = np.linspace(0.0001, 1, 101)
REPLICATES = 200
T_FINAL = 10000.0
STORED_TIME_POINTS = 1000
AVERAGE_FRACTION = 0.2
EXTINCTION_THRESHOLD = 1e-12
EFFICIENCY = 0.3
GROWTH_AUTOTROPH = 0.1
GROWTH_HETEROTROPH = GROWTH_AUTOTROPH / 3
SELF_REGULATION = 0.1
MAX_REPAIR_STEPS_PER_SPECIES = 100
SENSITIVITY_EFFICIENCIES = [0.3, 1.0]
SENSITIVITY_GROWTH_H = [-GROWTH_AUTOTROPH / 3, GROWTH_AUTOTROPH / 3, GROWTH_AUTOTROPH]
SI8_NETWORKS = 100
SI8_REMOVAL_REPLICATES = 10
SI8_REMOVED_SPECIES = 10

print('Runs per parameter pair:', len(RICHNESS) * REPLICATES)
print('Parameter pairs:', len(DEGREES) * len(STRENGTHS))
print('Integration:', (0, T_FINAL), '| biomass window starts at:', T_FINAL * (1 - AVERAGE_FRACTION))

## Model

In [ ]:
def matrix_min_degree_1(A, MT, conn, wei, N, rng):
    for _ in range(MAX_REPAIR_STEPS_PER_SPECIES * N):
        isolated = [i for i in range(N) if np.count_nonzero(A[i]) <= 1]
        if not isolated:
            return A
        ind = isolated[0]
        vect = wei * rng.uniform(-1, 1, N) * (rng.uniform(0, 1, N) < conn)
        rounds = 0
        while not np.any(vect):
            vect = wei * rng.uniform(-1, 1, N) * (rng.uniform(0, 1, N) < conn)
            rounds += 1
            if rounds > 1000:
                vect = wei * rng.uniform(-1, 1, N) * (rng.uniform(0, 1, N) < 1 / N)
            if rounds > 2000:
                raise RuntimeError('Could not repair an isolated node')
        MT[:, ind] = vect
        MT[ind, :] = -vect
        MT[1:, 0] = -np.abs(MT[1:, 0])
        MT[0, 1:] = np.abs(MT[0, 1:])
        MT = np.triu(MT)
        A = MT - MT.T + SELF_REGULATION * np.eye(N)
    raise RuntimeError(f'Network repair did not converge after '
                       f'{MAX_REPAIR_STEPS_PER_SPECIES * N} steps for N={N}, '
                       f'connection probability={conn:g}, strength={wei:g}')

def bfs_shortest_distances(adj_matrix, source_nodes):
    distances = [float('inf')] * len(adj_matrix)
    queue = deque(int(s) for s in source_nodes)
    for s in source_nodes:
        distances[s] = 0
    while queue:
        node = queue.popleft()
        for neighbor, connected in enumerate(adj_matrix[node]):
            if connected and distances[neighbor] == float('inf'):
                distances[neighbor] = distances[node] + 1
                queue.append(neighbor)
    return np.asarray(distances)

def make_network(N, degree, strength, rng, efficiency=EFFICIENCY):
    if strength <= 0:
        raise ValueError('Positive interaction strength is required to enforce at least one link per node')
    conn = min(degree / (N - 1), 1.0)
    MT = rng.uniform(-1, 1, (N, N)) * (rng.uniform(0, 1, (N, N)) < conn)
    MT[1:, 0] = -np.abs(MT[1:, 0])
    MT[0, 1:] = np.abs(MT[0, 1:])
    MT = strength * np.triu(MT)
    A = MT - MT.T + SELF_REGULATION * np.eye(N)
    A = matrix_min_degree_1(A, MT, conn, strength, N, rng)
    A[1:, 0] = -np.abs(A[1:, 0])
    A[0, 1:] = np.abs(A[0, 1:])
    A = efficiency * np.minimum(A, 0) + np.maximum(A, 0)
    off_diagonal = A.copy()
    np.fill_diagonal(off_diagonal, 0)
    if np.any(np.count_nonzero(off_diagonal, axis=1) == 0):
        raise RuntimeError('Network repair left an isolated species')
    source_nodes = np.where(np.sum(A > 0, axis=0) < 2)[0]
    layers = bfs_shortest_distances(A > 0, source_nodes)
    return A, layers

def solve(A, layers, t_final, growth_a=GROWTH_AUTOTROPH,
          growth_h=GROWTH_HETEROTROPH, x0=None, points=None):
    N = len(layers)
    growth = np.where(layers == 0, growth_a, growth_h)
    if x0 is None:
        x0 = np.full(N, 0.1)
    times = np.linspace(0, t_final, STORED_TIME_POINTS if points is None else points)
    result = solve_ivp(lambda t, x: x * (growth - A @ x), (0, t_final), x0,
                       method='RK45', t_eval=times, rtol=1e-6, atol=1e-9)
    if not result.success or not np.all(np.isfinite(result.y)):
        raise RuntimeError(result.message)
    return times, result.y

def summarize(N, degree, strength, rep, A, layers, times, states, scenario='screen',
              efficiency=EFFICIENCY, growth_h=GROWTH_HETEROTROPH):
    initial, final = states[:, 0], states[:, -1]
    cutoff = times[0] + (1 - AVERAGE_FRACTION) * (times[-1] - times[0])
    post_transient = times >= cutoff
    if np.count_nonzero(post_transient) < 2:
        raise ValueError('Need at least two stored times after the transient cutoff')
    time_mean = trapezoid(states[:, post_transient], times[post_transient], axis=1) / (
        times[post_transient][-1] - times[post_transient][0])
    bef_biomass = time_mean
    off_diagonal = A.copy()
    np.fill_diagonal(off_diagonal, 0)
    row = dict(N=N, degree=float(degree), strength=float(strength), replicate=rep,
               scenario=scenario, efficiency=efficiency, growth_h=growth_h,
               min_realized_degree=int(np.count_nonzero(off_diagonal, axis=1).min()),
               biomass_initial=float(initial.sum()), biomass_final=float(final.sum()),
               biomass_time_mean=float(time_mean.sum()), biomass_for_bef=float(bef_biomass.sum()),
               transient_cutoff=float(cutoff), post_transient_points=int(post_transient.sum()),
               richness_final=int(np.count_nonzero(final > EXTINCTION_THRESHOLD)),
               extinction_fraction=float(np.mean(final <= EXTINCTION_THRESHOLD)))
    for layer in range(6):
        mask = layers == layer if layer < 5 else layers > 4
        row[f'd{layer}'] = int(mask.sum())
        row[f'd{layer}_final'] = int(np.count_nonzero(final[mask] > EXTINCTION_THRESHOLD))
        row[f'md{layer}'] = float(bef_biomass[mask].sum())
    row['producer_fraction'] = row['md0'] / row['biomass_for_bef'] if row['biomass_for_bef'] > 0 else np.nan
    positive_interactions = np.maximum(A, 0).copy()
    np.fill_diagonal(positive_interactions, 0)
    row['energy_flow'] = float(final @ (positive_interactions @ final))
    return row


## Simulations

In [ ]:
rng = np.random.default_rng(SEED)
rows, trajectories, removals = [], [], []
for strength in STRENGTHS:
    for degree in DEGREES:
        for N in RICHNESS:
            for rep in range(REPLICATES):
                A, layers = make_network(N, degree, strength, rng)
                times, states = solve(A, layers, T_FINAL)
                rows.append(summarize(N, degree, strength, rep, A, layers, times, states))
                if rep == 0 and N == max(RICHNESS):
                    for i, t in enumerate(times):
                        trajectories.append(dict(degree=float(degree), strength=float(strength),
                                                 time=float(t), biomass=float(states[:, i].sum()),
                                                 producers=float(states[layers == 0, i].sum())))

simulations = pd.DataFrame(rows)
trajectories = pd.DataFrame(trajectories)
print('Screening table:', simulations.shape, '| trajectories:', trajectories.shape)
print('Runs per parameter pair:', simulations.groupby(['degree', 'strength']).size().agg(['min', 'max']).to_dict())
assert len(simulations) == len(RICHNESS)*len(DEGREES)*len(STRENGTHS)*REPLICATES
assert (simulations.min_realized_degree >= 1).all()
assert np.isfinite(simulations[['biomass_final', 'biomass_time_mean', 'biomass_for_bef']]).all().all()
assert (simulations[[f'd{i}' for i in range(6)]].sum(axis=1) == simulations.N).all()
assert (simulations[[f'd{i}_final' for i in range(6)]].sum(axis=1) == simulations.richness_final).all()
assert np.allclose(simulations[[f'md{i}' for i in range(6)]].sum(axis=1), simulations.biomass_for_bef)
print('Post-transient stored points:', simulations.post_transient_points.unique())
print(simulations[['N', 'degree', 'strength', 'biomass_final', 'biomass_time_mean',
                   'biomass_for_bef', 'richness_final']].head())

## Sensitivity and species removal

In [ ]:
sensitivity_rows, multiple_removals = [], []
sensitivity_rows.extend(rows)
for efficiency in SENSITIVITY_EFFICIENCIES:
    for growth_h in SENSITIVITY_GROWTH_H:
        if np.isclose(efficiency, EFFICIENCY) and np.isclose(growth_h, GROWTH_HETEROTROPH):
            continue
        for strength in STRENGTHS:
            for degree in DEGREES:
                for N in RICHNESS:
                    for rep in range(REPLICATES):
                        A, layers = make_network(N, degree, strength, rng, efficiency=efficiency)
                        times, states = solve(A, layers, T_FINAL, growth_h=growth_h)
                        sensitivity_rows.append(summarize(N, degree, strength, rep, A, layers,
                                                          times, states, scenario='sensitivity',
                                                          efficiency=efficiency, growth_h=growth_h))
for degree in sorted(set([float(min(DEGREES)), float(max(DEGREES))])):
    N = max(RICHNESS)
    strength = float(STRENGTHS[len(STRENGTHS)//2])
    A, layers = make_network(N, degree, strength, rng)
    times, states = solve(A, layers, T_FINAL)
    baseline = states[:, -1]
    target = int(np.argmax(baseline))
    removed = baseline.copy()
    removed[target] = 0
    A_deleted = A.copy()
    A_deleted[target, :] = 0
    A_deleted[:, target] = 0
    t2, y2 = solve(A_deleted, layers, T_FINAL, x0=removed)
    for t, value in zip(t2, y2.sum(axis=0)):
        removals.append(dict(degree=degree, strength=strength, N=N, replicate=0,
                             removed_count=1, time=float(t), biomass=float(value),
                             baseline_biomass=float(baseline.sum())))
N = 100
degree, strength = 5.0, 0.3
for network_id in range(SI8_NETWORKS):
    A, layers = make_network(N, degree, strength, rng)
    _, states = solve(A, layers, T_FINAL)
    baseline = states[:, -1]
    for removal_rep in range(SI8_REMOVAL_REPLICATES):
        deleted = rng.choice(N, SI8_REMOVED_SPECIES, replace=False)
        A_deleted = A.copy()
        A_deleted[deleted, :] = 0
        A_deleted[:, deleted] = 0
        x0 = baseline.copy()
        x0[deleted] = 0
        _, changed = solve(A_deleted, layers, T_FINAL, x0=x0)
        multiple_removals.append(dict(network_id=network_id, removal_replicate=removal_rep,
                                      N=N, degree=degree, strength=strength,
                                      removed_count=SI8_REMOVED_SPECIES,
                                      baseline_biomass=float(baseline.sum()),
                                      final_biomass=float(changed[:, -1].sum())))

removals = pd.DataFrame(removals)
sensitivity = pd.DataFrame(sensitivity_rows)
multiple_removals = pd.DataFrame(multiple_removals)
print('Removal observations:', len(removals), '| SI8 experiments:', len(multiple_removals))
print('Sensitivity settings:', sensitivity.groupby(['efficiency','growth_h']).size().to_dict()
      if len(sensitivity) else {})
if SAVE:
    OUTPUT_DIR.mkdir(exist_ok=True)
    simulations.to_csv(OUTPUT_DIR / 'simulations.csv', index=False)
    trajectories.to_csv(OUTPUT_DIR / 'trajectories.csv', index=False)
    removals.to_csv(OUTPUT_DIR / 'removals.csv', index=False)
    sensitivity.to_csv(OUTPUT_DIR / 'sensitivity.csv', index=False)
    multiple_removals.to_csv(OUTPUT_DIR / 'multiple_removals.csv', index=False)
    config = dict(profile=PROFILE, seed=SEED, richness=RICHNESS, degrees=list(map(float, DEGREES)),
                  strengths=list(map(float, STRENGTHS)), replicates=REPLICATES, t_final=T_FINAL,
                  source='manuscript simulation method',
                  biomass_for_bef='post-transient time average',
                  stored_time_points=STORED_TIME_POINTS, average_fraction=AVERAGE_FRACTION,
                  transient_cutoff=T_FINAL * (1 - AVERAGE_FRACTION),
                  connection_probability='degree/(N-1)',
                  network_isolation='local repair preserving link strengths',
                  max_repair_steps_per_species=MAX_REPAIR_STEPS_PER_SPECIES,
                  si8_networks=SI8_NETWORKS, si8_removal_replicates=SI8_REMOVAL_REPLICATES,
                  si8_removed_species=SI8_REMOVED_SPECIES)
    (OUTPUT_DIR / 'run_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
    print('Saved generated data in', OUTPUT_DIR.resolve())

## Saved data

The figure notebooks read the new tables in generated/.